[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/07_traing_problem_and%20soultion/01_gradient_problems/01_gradient_problems.ipynb)

# 01. Gradient Problems — Vanishing & Exploding Gradients

Full chain-rule analysis, numerical traces, and PyTorch fixes.

---


In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os, sys

if 'google.colab' in sys.modules:
    !git clone https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git
    os.chdir('Multimodal-Deep-Learning')
    os.chdir('07_traing_problem_and soultion/01_gradient_problems')
    !pip install -q torch torchvision matplotlib numpy
else:
    nb_dir = os.getcwd()
    if not os.path.basename(nb_dir) == '01_gradient_problems':
        os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..', '01_gradient_problems'))
    sys.path.append(os.path.join(os.getcwd(), '..', '..'))

print(f'Working directory: {os.getcwd()}')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11})


## 1. Why Gradients Fail — The Chain Rule Through $L$ Layers

For a network $h_0 \to h_1 \to \cdots \to h_L$ with loss $\mathcal{L}$:

$$
\frac{\partial \mathcal{L}}{\partial W_1} = \left(\prod_{l=1}^{L} \frac{\partial h_l}{\partial h_{l-1}}\right) \frac{\partial \mathcal{L}}{\partial h_L}
$$

Each Jacobian $J_l = \frac{\partial h_l}{\partial h_{l-1}}$ multiplies into the product. If $\lVert J_l \rVert < 1$ repeatedly → **vanishing**. If $\lVert J_l \rVert > 1$ → **exploding**.

```
Layer:   h0 --> h1 --> h2 --> ... --> hL --> L
Grad:    dL/dh0 <-- dL/dh1 <-- ... <-- dL/dhL
         product of L Jacobians (exponential scaling!)
```


### Numerical Example: 10 Layers, $\lVert J_l \rVert = 0.5$

| Layer $l$ | $\lVert \partial \mathcal{L}/\partial h_l \rVert$ | Cumulative product |
|-----------|-------------------------------------|-------------------|
| 10 | 1.0 | 1.0 |
| 9 | 0.5 | 0.5 |
| 8 | 0.25 | 0.25 |
| ... | ... | ... |
| 1 | $0.5^9 \approx 0.002$ | **vanishing** |

With $\lVert J_l \rVert = 1.2$: Layer 1 gradient $\approx 1.2^9 \approx 5.16$ → **exploding**.


In [ ]:
# Simulate gradient norm propagation through L layers
L = 10
norms_contract = [0.5 ** (L - l) for l in range(1, L + 1)]
norms_expand = [1.2 ** (L - l) for l in range(1, L + 1)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, L+1), norms_contract, color='steelblue')
axes[0].set_title('Vanishing: ||J||=0.5 per layer')
axes[0].set_xlabel('Layer'); axes[0].set_ylabel('||grad|| (relative)')
axes[1].bar(range(1, L+1), norms_expand, color='coral')
axes[1].set_title('Exploding: ||J||=1.2 per layer')
axes[1].set_xlabel('Layer')
plt.tight_layout(); plt.show()


## 2. Solution: Gradient Clipping

$$
g \leftarrow g \cdot \frac{\text{max\_norm}}{\max(\lVert g \rVert, \text{max\_norm})}
$$

If $\lVert g \rVert > \text{max\_norm}$, scale down; otherwise unchanged. Preserves direction, caps magnitude.


In [ ]:
def clip_grad_norm_manual(grad, max_norm=1.0):
    norm = grad.norm()
    if norm > max_norm:
        return grad * (max_norm / norm)
    return grad

g = torch.tensor([3.0, 4.0])  # ||g|| = 5
g_clipped = clip_grad_norm_manual(g, max_norm=1.0)
print(f'Original ||g||={g.norm():.2f}, clipped ||g||={g_clipped.norm():.4f}')
print(f'Clipped vector: {g_clipped.tolist()}')


## 3. Solution: Residual Connections

$$
h_l = F(h_{l-1}) + h_{l-1}
$$

**Gradient flow proof:**

$$
\frac{\partial h_L}{\partial h_l} = \frac{\partial h_L}{\partial h_{l+1}}\left(I + \frac{\partial F(h_l)}{\partial h_l}\right)
$$

The identity term $I$ provides a **direct path** for gradients — even if $\partial F/\partial h_l \approx 0$, gradients can flow via the skip connection.


In [ ]:
class PlainDeepNet(nn.Module):
    def __init__(self, depth=10, dim=32):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, dim), nn.Tanh()) for _ in range(depth)
        ])
        self.head = nn.Linear(dim, 1)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.head(x).squeeze(-1)

class ResidualDeepNet(nn.Module):
    def __init__(self, depth=10, dim=32):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, dim), nn.Tanh()) for _ in range(depth)
        ])
        self.head = nn.Linear(dim, 1)

    def forward(self, x):
        for layer in self.layers:
            x = x + layer(x)
        return self.head(x).squeeze(-1)

def grad_norms(model, x, y):
    model.zero_grad()
    loss = F.mse_loss(model(x), y)
    loss.backward()
    norms = []
    for layer in model.layers:
        norms.append(layer[0].weight.grad.norm().item())
    return norms

x = torch.randn(16, 32); y = torch.randn(16)
plain = PlainDeepNet(); res = ResidualDeepNet()
n_plain = grad_norms(plain, x, y)
n_res = grad_norms(res, x, y)

plt.plot(range(1, 11), n_plain, 'o-', label='Plain (vanishing)'); plt.plot(range(1, 11), n_res, 's-', label='Residual')
plt.xlabel('Layer'); plt.ylabel('Grad norm'); plt.legend(); plt.title('Gradient Norms: Plain vs Residual'); plt.show()


## 4. Layer Normalization

$$
\text{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta
$$

Normalizes activations per sample → stabilizes Jacobian singular values, reducing internal covariate shift.


## 5. Proper Initialization — Xavier & He

**Xavier (Glorot):** $\text{Var}(W) = \frac{2}{n_{in} + n_{out}}$ for tanh/sigmoid.

**He (Kaiming):** $\text{Var}(W) = \frac{2}{n_{in}}$ for ReLU (accounts for halving variance through ReLU).

PyTorch: `nn.init.xavier_uniform_`, `nn.init.kaiming_normal_(..., nonlinearity='relu')`.


In [ ]:
def compare_inits(n_in=256, n_out=256, n_trials=100):
    results = {}
    for name, init_fn in [('Xavier', lambda w: nn.init.xavier_uniform_(w)),
                          ('Default', lambda w: None),
                          ('He', lambda w: nn.init.kaiming_normal_(w, nonlinearity='relu'))]:
        vars_ = []
        for _ in range(n_trials):
            w = torch.empty(n_in, n_out)
            if name != 'Default': init_fn(w)
            x = torch.randn(64, n_in)
            h = F.relu(x @ w)
            vars_.append(h.var().item())
        results[name] = np.mean(vars_)
    return results

r = compare_inits()
plt.bar(r.keys(), r.values()); plt.ylabel('Mean activation variance'); plt.title('Init schemes'); plt.show()


## 6. Mixed Precision — Gradient Scaling

FP16 gradients can underflow to zero. **Loss scaling:** multiply loss by $S$, backprop, then unscale gradients before optimizer step.

```
loss_scaled = loss * scale
loss_scaled.backward()
grads = grads / scale  # restore magnitude
```


In [ ]:
# Demonstrate FP16 underflow without scaling
small_grad = torch.tensor(1e-8, dtype=torch.float16)
print(f'FP16 grad stored as: {small_grad.item()} (underflow to 0!)')
scale = 1024.0
scaled = (torch.tensor(1e-8) * scale).float()
print(f'With loss scaling x{scale:.0f}: effective grad = {scaled.item():.2e}')


## 7. Decision Table — When to Use Each Solution

| Symptom | Likely Cause | Fix | Priority |
|---------|--------------|-----|----------|
| Early layers grad ≈ 0 | Vanishing | Residual + He init + LN | High |
| Loss spikes, grad > 100 | Exploding | Clip grad (max_norm=1.0) | Immediate |
| Activations drift over epochs | Internal covariate shift | LayerNorm / BatchNorm | Medium |
| FP16 training, no updates | Grad underflow | Loss scaling (AMP) | High |
| Very deep MLP/RNN | Product of Jacobians | Residual + skip connections | High |


## 8. End-to-End Demo: 10-Layer Network Before/After Fixes


In [ ]:
def train_and_track(model, steps=50):
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    x = torch.randn(32, 32); y = torch.randn(32)
    history = []
    for _ in range(steps):
        opt.zero_grad()
        loss = F.mse_loss(model(x), y)
        loss.backward()
        gn = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
        history.append(gn)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    return history

h1 = train_and_track(PlainDeepNet())
h2 = train_and_track(ResidualDeepNet())
plt.plot(h1, label='Plain + clipping'); plt.plot(h2, label='Residual + clipping')
plt.xlabel('Step'); plt.ylabel('Total grad norm'); plt.legend(); plt.title('Training stability'); plt.show()


## References & Further Reading

- Glorot & Bengio (2010) — Understanding the difficulty of training deep feedforward neural networks — [arXiv:1006.2785](https://arxiv.org/abs/1006.2785)
- He et al. (2015) — Delving Deep into Rectifiers — [arXiv:1502.01852](https://arxiv.org/abs/1502.01852)
- Ba et al. (2016) — Layer Normalization — [arXiv:1607.06450](https://arxiv.org/abs/1607.06450)

**Blog posts:**
- [Lilian Weng — Why ResNet Works](https://lilianweng.github.io/posts/2017-06-08-overview/)
- [Jay Alammar — Visualizing Learning Rates](https://jalammar.github.io/visualizing-learning-rate/)
